# 情報疲れ調査の探索分析

`survey/*.csv` を読み込み、local university における情報疲れの予備調査として、媒体別・負担理由別・媒体と負担理由の関係を確認する notebook です。

主な確認観点:

- SNS と AI チャット・AI 検索が、負担を感じる媒体として多いか
- 負担理由は、情報量・終わりのなさ・理解の難しさのどれに寄っているか
- 現行設問で検証できることと、次回調査で追加すべき設問を分ける

## 1. CSV の読み込み

標準ライブラリだけで動くようにしています。`matplotlib` が入っていればグラフも描画し、入っていなければ表と ASCII バーで確認できます。

In [1]:
from pathlib import Path
import csv
from collections import Counter, defaultdict

ROOT = Path.cwd()
SURVEY_DIR = ROOT / 'survey' if (ROOT / 'survey').is_dir() else ROOT
csv_paths = sorted(SURVEY_DIR.glob('survey-*.csv'))

assert csv_paths, f'No survey CSV files found under {SURVEY_DIR}'

rows = []
for path in csv_paths:
    with path.open(encoding='utf-8-sig', newline='') as f:
        for row in csv.DictReader(f):
            row['_file'] = path.name
            rows.append(row)

valid_rows = [r for r in rows if r.get('失敗フラグ') != '1']
failed_rows = [r for r in rows if r.get('失敗フラグ') == '1']

print('CSV files')
for path in csv_paths:
    print('-', path.name)
print()
print(f'total rows: {len(rows)}')
print(f'valid rows: {len(valid_rows)}')
print(f'failed rows: {len(failed_rows)}')

CSV files
- survey-2026-05-01-ikejiri.csv
- survey-2026-05-01-kitada.csv
- survey-2026-05-01-miyao.csv

total rows: 24
valid rows: 23
failed rows: 1


## 2. 集計ヘルパー

In [2]:
MISSING = '(未回答)'

def value(row, column):
    return (row.get(column) or '').strip() or MISSING

def pct(n, total):
    return n / total * 100 if total else 0

def print_table(headers, body):
    widths = [len(str(h)) for h in headers]
    for row in body:
        for i, cell in enumerate(row):
            widths[i] = max(widths[i], len(str(cell)))
    fmt = ' | '.join('{:<%d}' % w for w in widths)
    print(fmt.format(*headers))
    print('-+-'.join('-' * w for w in widths))
    for row in body:
        print(fmt.format(*row))

def count_table(title, counter, total):
    print('\n' + title)
    body = [[k, v, f'{pct(v, total):.1f}%'] for k, v in counter.most_common()]
    print_table(['category', 'n', 'share'], body)

def ascii_bar(counter, total, width=32):
    max_n = max(counter.values(), default=1)
    for label, n in counter.most_common():
        blocks = max(1, round(n / max_n * width)) if n else 0
        bar = '█' * blocks
        print(f'{label:<24} {bar} {n} ({pct(n, total):.1f}%)')

## 3. 基本集計

`失敗フラグ=1` を除外した有効回答だけで集計します。

In [3]:
n_valid = len(valid_rows)

by_file = Counter(value(r, '_file') for r in valid_rows)
media_counts = Counter(value(r, '媒体') for r in valid_rows)
burden_counts = Counter(value(r, '負担のタイプ') for r in valid_rows)
gender_counts = Counter(value(r, '性別') for r in valid_rows)

count_table('有効回答数 by file', by_file, n_valid)
count_table('媒体', media_counts, n_valid)
count_table('負担のタイプ', burden_counts, n_valid)
count_table('性別', gender_counts, n_valid)


有効回答数 by file
category                      | n | share
------------------------------+---+------
survey-2026-05-01-ikejiri.csv | 8 | 34.8%
survey-2026-05-01-miyao.csv   | 8 | 34.8%
survey-2026-05-01-kitada.csv  | 7 | 30.4%

媒体
category      | n | share
--------------+---+------
SNS           | 9 | 39.1%
AIチャット・AI検索   | 4 | 17.4%
メール・チャット・通知   | 3 | 13.0%
新聞・雑誌         | 2 | 8.7% 
動画プラットフォーム    | 2 | 8.7% 
特にない / 答えたくない | 1 | 4.3% 
その他           | 1 | 4.3% 
テレビ           | 1 | 4.3% 

負担のタイプ
category                | n | share
------------------------+---+------
情報量が多く、追いきれない           | 9 | 39.1%
調べても次々に情報が出てきて、終わりが見えない | 7 | 30.4%
内容が難しく、理解に時間がかかる        | 5 | 21.7%
選択肢が多く、選ぶのに迷う           | 1 | 4.3% 
その他                     | 1 | 4.3% 

性別
category | n  | share
---------+----+------
男性       | 18 | 78.3%
女性       | 4  | 17.4%
(未回答)    | 1  | 4.3% 


## 4. 可視化ヘルパー

カテゴリ比較は横棒グラフが読みやすいです。`matplotlib` が使えない環境では ASCII バーを出します。

In [4]:
try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ModuleNotFoundError:
    HAS_MPL = False

COLORS = ['#2563eb', '#16a34a', '#dc2626', '#9333ea', '#f59e0b', '#0891b2', '#64748b']

def plot_bar(counter, title, total):
    if not HAS_MPL:
        print(title)
        ascii_bar(counter, total)
        return
    items = counter.most_common()
    labels = [k for k, _ in items]
    values = [v for _, v in items]
    fig, ax = plt.subplots(figsize=(9, max(3, 0.45 * len(items))))
    bars = ax.barh(labels, values, color='#2563eb')
    ax.invert_yaxis()
    ax.set_title(title)
    ax.set_xlabel('有効回答数')
    ax.bar_label(bars, labels=[f'{v} ({pct(v, total):.1f}%)' for v in values], padding=4)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    plt.tight_layout()
    plt.show()

def plot_stacked(crosstab, media_order, burden_order):
    if not HAS_MPL:
        print('matplotlib is not available; use the cross-tab table above.')
        return
    fig, ax = plt.subplots(figsize=(11, max(3, 0.45 * len(media_order))))
    left = [0] * len(media_order)
    for i, burden in enumerate(burden_order):
        values = [crosstab[m].get(burden, 0) for m in media_order]
        ax.barh(media_order, values, left=left, label=burden, color=COLORS[i % len(COLORS)])
        left = [a + b for a, b in zip(left, values)]
    ax.invert_yaxis()
    ax.set_title('媒体別の負担理由')
    ax.set_xlabel('有効回答数')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    plt.tight_layout()
    plt.show()

print(f'matplotlib available: {HAS_MPL}')

matplotlib available: False


## 5. 媒体別・負担理由別の可視化

In [5]:
plot_bar(media_counts, '最も負担・疲労感を感じる媒体', n_valid)
plot_bar(burden_counts, '負担・疲労感に最も近い理由', n_valid)

最も負担・疲労感を感じる媒体
SNS                      ████████████████████████████████ 9 (39.1%)
AIチャット・AI検索              ██████████████ 4 (17.4%)
メール・チャット・通知              ███████████ 3 (13.0%)
新聞・雑誌                    ███████ 2 (8.7%)
動画プラットフォーム               ███████ 2 (8.7%)
特にない / 答えたくない            ████ 1 (4.3%)
その他                      ████ 1 (4.3%)
テレビ                      ████ 1 (4.3%)
負担・疲労感に最も近い理由
情報量が多く、追いきれない            ████████████████████████████████ 9 (39.1%)
調べても次々に情報が出てきて、終わりが見えない  █████████████████████████ 7 (30.4%)
内容が難しく、理解に時間がかかる         ██████████████████ 5 (21.7%)
選択肢が多く、選ぶのに迷う            ████ 1 (4.3%)
その他                      ████ 1 (4.3%)


## 6. 媒体 × 負担理由

SNS と AI チャット・AI 検索で、負担理由の出方が違うかを確認します。件数が少ないため、比率だけでなく生の件数を見るのが重要です。

In [6]:
media_order = [k for k, _ in media_counts.most_common()]
burden_order = [k for k, _ in burden_counts.most_common()]

crosstab = {media: Counter() for media in media_order}
for row in valid_rows:
    crosstab[value(row, '媒体')][value(row, '負担のタイプ')] += 1

body = []
for media in media_order:
    values = [crosstab[media].get(burden, 0) for burden in burden_order]
    body.append([media] + values + [sum(values)])

print_table(['媒体'] + burden_order + ['total'], body)
plot_stacked(crosstab, media_order, burden_order)

媒体            | 情報量が多く、追いきれない | 調べても次々に情報が出てきて、終わりが見えない | 内容が難しく、理解に時間がかかる | 選択肢が多く、選ぶのに迷う | その他 | total
--------------+---------------+-------------------------+------------------+---------------+-----+------
SNS           | 5             | 2                       | 1                | 0             | 1   | 9    
AIチャット・AI検索   | 1             | 2                       | 0                | 1             | 0   | 4    
メール・チャット・通知   | 1             | 1                       | 1                | 0             | 0   | 3    
新聞・雑誌         | 1             | 0                       | 1                | 0             | 0   | 2    
動画プラットフォーム    | 0             | 2                       | 0                | 0             | 0   | 2    
特にない / 答えたくない | 0             | 0                       | 1                | 0             | 0   | 1    
その他           | 1             | 0                       | 0                | 0             | 0   | 1    
テレビ           | 0             | 0                      

## 7. 仮説に対する主要指標

現在の設問では「情報疲れが発生しているか」の母集団内発生率は直接測れません。一方で、「負担媒体として SNS と AI チャット・AI 検索が多いか」は確認できます。

In [7]:
target_media = ['SNS', 'AIチャット・AI検索']
target_n = sum(media_counts.get(m, 0) for m in target_media)
sns_n = media_counts.get('SNS', 0)
ai_n = media_counts.get('AIチャット・AI検索', 0)

print('Hypothesis check')
print(f'SNS: {sns_n} / {n_valid} = {pct(sns_n, n_valid):.1f}%')
print(f'AIチャット・AI検索: {ai_n} / {n_valid} = {pct(ai_n, n_valid):.1f}%')
print(f'SNS + AIチャット・AI検索: {target_n} / {n_valid} = {pct(target_n, n_valid):.1f}%')
print()
print('Top media')
for rank, (media, n) in enumerate(media_counts.most_common(), start=1):
    print(f'{rank}. {media}: {n} ({pct(n, n_valid):.1f}%)')

Hypothesis check
SNS: 9 / 23 = 39.1%
AIチャット・AI検索: 4 / 23 = 17.4%
SNS + AIチャット・AI検索: 13 / 23 = 56.5%

Top media
1. SNS: 9 (39.1%)
2. AIチャット・AI検索: 4 (17.4%)
3. メール・チャット・通知: 3 (13.0%)
4. 新聞・雑誌: 2 (8.7%)
5. 動画プラットフォーム: 2 (8.7%)
6. 特にない / 答えたくない: 1 (4.3%)
7. その他: 1 (4.3%)
8. テレビ: 1 (4.3%)


## 8. 自由記述・失敗回答の確認

自由記述は件数が少なくても、次回調査の選択肢追加や設問修正に使えます。

In [8]:
free_text_rows = []
for row in valid_rows:
    media_free = value(row, '媒体_自由記述')
    burden_free = value(row, '負担のタイプ_自由記述')
    if media_free != MISSING or burden_free != MISSING:
        free_text_rows.append([
            value(row, '_file'),
            value(row, '媒体'),
            '' if media_free == MISSING else media_free,
            value(row, '負担のタイプ'),
            '' if burden_free == MISSING else burden_free,
        ])

print('自由記述')
if free_text_rows:
    print_table(['file', '媒体', '媒体_自由記述', '負担のタイプ', '負担のタイプ_自由記述'], free_text_rows)
else:
    print('none')

print('\n失敗回答')
if failed_rows:
    failed_body = [[value(r, '_file'), value(r, 'timestamp'), value(r, '媒体'), value(r, '負担のタイプ'), value(r, '性別')] for r in failed_rows]
    print_table(['file', 'timestamp', '媒体', '負担のタイプ', '性別'], failed_body)
else:
    print('none')

自由記述
file                         | 媒体  | 媒体_自由記述 | 負担のタイプ        | 負担のタイプ_自由記述
-----------------------------+-----+---------+---------------+------------
survey-2026-05-01-kitada.csv | その他 | 人       | 情報量が多く、追いきれない |            
survey-2026-05-01-miyao.csv  | SNS |         | その他           | AIかどうかわからない

失敗回答
file                         | timestamp                | 媒体  | 負担のタイプ | 性別
-----------------------------+--------------------------+-----+--------+---
survey-2026-05-01-kitada.csv | 2026-05-01T07:31:49.426Z | SNS | 答えたくない | 男性


## 9. 読み取りメモ

- 現行データからは、負担媒体として SNS が最も多いか、AI チャット・AI 検索が上位に出るかを確認する。
- SNS と AI チャット・AI 検索の合算割合は、仮説を説明する主指標にできる。
- ただし、現行設問は負担・疲労感がある前提で「最も負担を感じる媒体」を聞いているため、local university 全体で情報疲れがどの程度発生しているかは直接検証できない。
- 次回調査では、最初に「最近1か月、情報接触によって疲れ・負担を感じることがありますか」を 5 段階で聞くと、発生頻度を検証できる。
- 媒体利用頻度も聞くと、SNS や AI が多い理由が「利用者が多いから」なのか「負担になりやすいから」なのかを分けやすい。